The models have been trained on images from

https://imagenet.standford.edu

with a heirarchy of labels that comes from

https://wordnet.princeton.edu

The pretrained models will come from the TorchVision project

https://github.com/pytorch/vision

### Original Model Sources

AlexNet - htps://mng.bz/1o6z 

ResNet - https://arxiv.org/pdf/1512.03385.pdf

Inception v3 - https://arxiv.org/pdf/1512.00567.pdf

In [ ]:
from torchvision import models

In [ ]:
dir(models)

In [ ]:
len(dir(models))

Note that names with capital letters are classes and the uncapitalized versions are instances of those models, e.g. with different weights or number of layers

In [ ]:
alexnet = models.AlexNet() # this is the base architecture with randomized initial weights

In [ ]:
# resnet101 is a 101-layer residual network
resnet = models.resnet101(pretrained=True)

In [ ]:
resnet

In [ ]:
from torchvision import transforms # module to help modify image inputs to the expected image input format
preprocess = transforms.Compose([
    transforms.Resize(256), # square resize, hence one value for 256x256
    transforms.CenterCrop(224), # crop to 224x224 around the center
    transforms.ToTensor(), 
    transforms.Normalize( # fixed to what was used during training. Probably look at original paper to find these?
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
        )
    ])

We will use Pillow library for image manipulation

In [ ]:
from PIL import Image
img = Image.open("../../data/p1ch2/bobby.jpg")

In [ ]:
img

In [ ]:
# get the image formated for the model
img_t = preprocess(img)

In [ ]:
import torch
# unexplained at this point in the book
batch_t = torch.unsqueeze(img_t,0)

In [ ]:
# inference = prediction

# set the model to evaluation mode to avoid some processes like batch normalization and dropout which provide meaningless outputs
resnet.eval()

In [ ]:
out = resnet(batch_t) # these should be the values that get fed into, say, softmax
out

In [ ]:
# we still need the labels that are associated with these predictions
with open('../../data/p1ch2/imagenet_classes.txt') as f:
    labels = [line.strip() for line in f.readlines()] # list comprehension to read each line, strip leading and trailing whitespace, and store in list

In [ ]:
labels

In [ ]:
# we don't need to bother with getting the probabilities/confidence values since, e.g. softmax is strictly monotonic.
# Ranking the inputs to softmax is sufficient. Hence, max confidence prediction is max value from the above output
_, index = torch.max(out,1) # technically a 2d array, but really a row vector. Max across the columns, i.e. along the row

In [ ]:
labels[index]

In [ ]:
percentage = torch.nn.functional.softmax(out, dim=1)[0] *100 # grab data, hence [0], and multiply by 100 for percentage
labels[index[0]] , percentage[index[0]].item() # the indexing of index is not needed here but perhaps is good practice later?

In [ ]:
# to get a ranked ordering of the predictions along with the confidence
_, indices = torch.sort(out, descending=True) # the sorting function returns the values along with the original index values reordered
[(labels[idx], percentage[idx].item()) for idx in indices[0][:5]]